# Notebook 06 — Seller ESG Modeling (9 Features)

## Mục tiêu

Notebook này thực hiện:

1. Đọc Seller-level ESG dataset gồm **9 feature ESG**
2. Nhập trực tiếp **trọng số AHP đã tính sẵn**
3. Chuẩn hóa và đảo chiều các biến khiếu nại
4. Tính `ESG_reference_score`
5. Huấn luyện và so sánh 5 mô hình:
   - Linear Regression
   - Ridge Regression
   - XGBoost Regressor
   - Voting Regressor
   - Stacking Regressor
6. Đánh giá bằng MAE, MSE, RMSE, R² và 5-fold CV
7. Phân tích permutation importance và lưu model

> Phiên bản hiện tại tạm thời sử dụng 9 feature và chưa đưa
> `packaging_complaint_count` vào mô hình.

> Lưu ý phương pháp: `ESG_reference_score` là nhãn tham chiếu
> được tạo từ trọng số AHP và chính các feature đầu vào.
> Vì vậy Linear Regression có thể đạt R² xấp xỉ 1.
> Kết quả này phản ánh khả năng tái tạo công thức chấm điểm,
> không phải xác thực độc lập.


In [34]:
from pathlib import Path
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "Thiếu xgboost. Cài bằng: pip install xgboost"
    ) from exc

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


In [35]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "Dataset" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "Dataset" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "ESG_Modeling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_DATASET_PATH = INPUT_DIR / "seller_esg_feature_dataset.csv"

REFERENCE_OUTPUT_PATH = OUTPUT_DIR / "seller_esg_reference_dataset.csv"
WEIGHT_OUTPUT_PATH = OUTPUT_DIR / "ahp_fixed_weights.csv"
PERFORMANCE_OUTPUT_PATH = OUTPUT_DIR / "model_performance.csv"
IMPORTANCE_OUTPUT_PATH = OUTPUT_DIR / "feature_importance.csv"
MODEL_OUTPUT_PATH = OUTPUT_DIR / "seller_esg_model.pkl"
METADATA_OUTPUT_PATH = OUTPUT_DIR / "model_metadata.json"

print("Input:", FEATURE_DATASET_PATH)
print("Output:", OUTPUT_DIR)


Input: d:\UNI\NCKH\CTD2026_DT049\Dataset\processed\seller_esg_feature_dataset.csv
Output: d:\UNI\NCKH\CTD2026_DT049\ESG_Modeling


In [36]:
if not FEATURE_DATASET_PATH.exists():
    candidates = list(PROJECT_ROOT.rglob("seller_esg_feature_dataset.csv"))
    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy {FEATURE_DATASET_PATH.name}"
        )
    FEATURE_DATASET_PATH = candidates[0]

seller_esg_df = pd.read_csv(FEATURE_DATASET_PATH)

print("Loaded:", FEATURE_DATASET_PATH)
print("Shape:", seller_esg_df.shape)
display(seller_esg_df.head())


Loaded: d:\UNI\NCKH\CTD2026_DT049\Dataset\processed\seller_esg_feature_dataset.csv
Shape: (261, 10)


,seller_name,environmental_keyword_count,sustainable_material_count,eco_label_count,product_quality_complaint_count,product_damage_complaint_count,product_safety_complaint_count,customer_service_complaint_count,counterfeit_complaint_count,governance_keyword_count
0,101Dealz,0.0,1.666667,0.0,0.0,0.033333,0.066667,0.0,0.0,0.5
1,123(EXOPRT WAREHOUSE),0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0
2,24 Caret,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.1
3,"33,000ft outdoor",0.0,1.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0
4,5Mayi Official,0.0,1.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0


In [37]:
SELLER_COLUMN_CANDIDATES = [
    "seller_name",
    "seller",
    "seller_id",
    "shop_name",
    "shop_id",
]

seller_column = next(
    (
        column
        for column in SELLER_COLUMN_CANDIDATES
        if column in seller_esg_df.columns
    ),
    None,
)

if seller_column is None:
    raise ValueError("Không tìm thấy cột định danh seller.")

print("Seller column:", seller_column)


Seller column: seller_name


In [38]:
# ============================================================
# 9 ESG FEATURES
# ============================================================

indicator_mapping = pd.DataFrame(
    [
        {
            "feature": "environmental_keyword_count",
            "indicator": "Environmental Commitment",
            "dimension": "Environmental",
            "dimension_code": "E",
            "direction": "positive",
        },
        {
            "feature": "sustainable_material_count",
            "indicator": "Sustainable Material Adoption",
            "dimension": "Environmental",
            "dimension_code": "E",
            "direction": "positive",
        },
        {
            "feature": "eco_label_count",
            "indicator": "Eco Label Adoption",
            "dimension": "Environmental",
            "dimension_code": "E",
            "direction": "positive",
        },
        {
            "feature": "product_quality_complaint_count",
            "indicator": "Product Quality Responsibility",
            "dimension": "Social",
            "dimension_code": "S",
            "direction": "negative",
        },
        {
            "feature": "product_damage_complaint_count",
            "indicator": "Product Durability",
            "dimension": "Social",
            "dimension_code": "S",
            "direction": "negative",
        },
        {
            "feature": "product_safety_complaint_count",
            "indicator": "Product Safety Responsibility",
            "dimension": "Social",
            "dimension_code": "S",
            "direction": "negative",
        },
        {
            "feature": "customer_service_complaint_count",
            "indicator": "Customer Relationship Management",
            "dimension": "Governance",
            "dimension_code": "G",
            "direction": "negative",
        },
        {
            "feature": "counterfeit_complaint_count",
            "indicator": "Business Integrity",
            "dimension": "Governance",
            "dimension_code": "G",
            "direction": "negative",
        },
        {
            "feature": "governance_keyword_count",
            "indicator": "Transparency & Responsibility",
            "dimension": "Governance",
            "dimension_code": "G",
            "direction": "positive",
        },
    ]
)

esg_features = (
    indicator_mapping[
        "feature"
    ]
    .tolist()
)

negative_direction_features = (
    indicator_mapping.loc[
        indicator_mapping[
            "direction"
        ] == "negative",
        "feature",
    ]
    .tolist()
)

print(
    "Number of ESG features:",
    len(esg_features),
)

display(
    indicator_mapping
)


Number of ESG features: 9


,feature,indicator,dimension,dimension_code,direction
0,environmental_keyword_count,Environmental Commitment,Environmental,E,positive
1,sustainable_material_count,Sustainable Material Adoption,Environmental,E,positive
2,eco_label_count,Eco Label Adoption,Environmental,E,positive
3,product_quality_complaint_count,Product Quality Responsibility,Social,S,negative
4,product_damage_complaint_count,Product Durability,Social,S,negative
5,product_safety_complaint_count,Product Safety Responsibility,Social,S,negative
6,customer_service_complaint_count,Customer Relationship Management,Governance,G,negative
7,counterfeit_complaint_count,Business Integrity,Governance,G,negative
8,governance_keyword_count,Transparency & Responsibility,Governance,G,positive


In [39]:
# ============================================================
# KIỂM TRA DATASET ĐÃ CÓ ĐỦ 9 FEATURE CHƯA
# ============================================================

missing_features = [
    feature
    for feature in esg_features
    if feature
    not in seller_esg_df.columns
]

if missing_features:
    raise ValueError(
        "Seller-level dataset chưa đủ 9 feature.\n"
        "Thiếu các cột:\n- "
        + "\n- ".join(
            missing_features
        )
        + "\n\nHãy kiểm tra lại file "
          "seller_esg_feature_dataset.csv."
    )

print(
    "Đã có đủ 9 ESG features."
)


Đã có đủ 9 ESG features.


In [40]:
# ============================================================
# NHẬP TRỌNG SỐ AHP ĐÃ TÍNH SẴN — 9 FEATURES
# ============================================================
# Các trọng số dưới đây được lấy từ kết quả AHP đã chốt
# trong tài liệu nghiên cứu.
#
# Notebook 06 không tính lại ma trận so sánh cặp,
# eigenvector, CI hoặc CR.

AHP_GLOBAL_WEIGHTS_INPUT = {
    "environmental_keyword_count": 0.0683,
    "sustainable_material_count": 0.4072,
    "eco_label_count": 0.1675,

    "product_quality_complaint_count": 0.0737,
    "product_damage_complaint_count": 0.0301,
    "product_safety_complaint_count": 0.1792,

    "customer_service_complaint_count": 0.0079,
    "counterfeit_complaint_count": 0.0469,
    "governance_keyword_count": 0.0193,
}

if (
    set(
        AHP_GLOBAL_WEIGHTS_INPUT
    )
    != set(
        esg_features
    )
):
    missing_in_weights = sorted(
        set(
            esg_features
        )
        - set(
            AHP_GLOBAL_WEIGHTS_INPUT
        )
    )

    extra_in_weights = sorted(
        set(
            AHP_GLOBAL_WEIGHTS_INPUT
        )
        - set(
            esg_features
        )
    )

    raise ValueError(
        "Weight-feature mismatch. "
        f"Missing={missing_in_weights}; "
        f"Extra={extra_in_weights}"
    )

if any(
    weight < 0
    for weight
    in AHP_GLOBAL_WEIGHTS_INPUT.values()
):
    raise ValueError(
        "Trọng số AHP không được âm."
    )

raw_weight_sum = sum(
    AHP_GLOBAL_WEIGHTS_INPUT.values()
)

if raw_weight_sum <= 0:
    raise ValueError(
        "Tổng trọng số AHP phải lớn hơn 0."
    )

# Chuẩn hóa nhẹ vì các trọng số trong báo cáo
# đã được làm tròn đến bốn chữ số thập phân.
AHP_GLOBAL_WEIGHTS = {
    feature: (
        weight
        / raw_weight_sum
    )
    for feature, weight
    in AHP_GLOBAL_WEIGHTS_INPUT.items()
}

print(
    "Raw AHP weight sum:",
    raw_weight_sum,
)

print(
    "Normalized AHP weight sum:",
    sum(
        AHP_GLOBAL_WEIGHTS.values()
    ),
)


Raw AHP weight sum: 1.0001000000000002
Normalized AHP weight sum: 0.9999999999999999


In [41]:
# Bảng trọng số dùng trong scoring

weight_df = indicator_mapping.copy()
weight_df["raw_global_weight"] = weight_df["feature"].map(
    AHP_GLOBAL_WEIGHTS_INPUT
)
weight_df["global_weight"] = weight_df["feature"].map(
    AHP_GLOBAL_WEIGHTS
)

dimension_weight_df = (
    weight_df.groupby(
        ["dimension_code", "dimension"],
        as_index=False,
    )["global_weight"]
    .sum()
    .rename(columns={"global_weight": "dimension_weight"})
)

weight_df = weight_df.merge(
    dimension_weight_df,
    on=["dimension_code", "dimension"],
    how="left",
)

weight_df["local_weight_within_dimension"] = (
    weight_df["global_weight"]
    / weight_df["dimension_weight"]
)

display(weight_df)
display(dimension_weight_df)

weight_df.to_csv(
    WEIGHT_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)


,feature,indicator,dimension,dimension_code,direction,raw_global_weight,global_weight,dimension_weight,local_weight_within_dimension
0,environmental_keyword_count,Environmental Commitment,Environmental,E,positive,0.0683,0.068293,0.642936,0.106221
1,sustainable_material_count,Sustainable Material Adoption,Environmental,E,positive,0.4072,0.407159,0.642936,0.633281
2,eco_label_count,Eco Label Adoption,Environmental,E,positive,0.1675,0.167483,0.642936,0.260498
3,product_quality_complaint_count,Product Quality Responsibility,Social,S,negative,0.0737,0.073693,0.282972,0.260424
4,product_damage_complaint_count,Product Durability,Social,S,negative,0.0301,0.030097,0.282972,0.106360
5,product_safety_complaint_count,Product Safety Responsibility,Social,S,negative,0.1792,0.179182,0.282972,0.633216
6,customer_service_complaint_count,Customer Relationship Management,Governance,G,negative,0.0079,0.007899,0.074093,0.106613
7,counterfeit_complaint_count,Business Integrity,Governance,G,negative,0.0469,0.046895,0.074093,0.632928
8,governance_keyword_count,Transparency & Responsibility,Governance,G,positive,0.0193,0.019298,0.074093,0.260459


,dimension_code,dimension,dimension_weight
0,E,Environmental,0.642936
1,G,Governance,0.074093
2,S,Social,0.282972


In [42]:
# ============================================================
# NUMERIC CONVERSION, IMPUTATION, MIN-MAX NORMALIZATION
# ============================================================

indicator_data = seller_esg_df[
    [seller_column] + esg_features
].copy()

for feature in esg_features:
    indicator_data[feature] = pd.to_numeric(
        indicator_data[feature],
        errors="coerce",
    )

imputer = SimpleImputer(strategy="median")
imputed_values = imputer.fit_transform(
    indicator_data[esg_features]
)

imputed_df = pd.DataFrame(
    imputed_values,
    columns=esg_features,
    index=indicator_data.index,
)

scaler = MinMaxScaler()
normalized_values = scaler.fit_transform(imputed_df)

reference_df = pd.DataFrame(
    normalized_values,
    columns=esg_features,
    index=indicator_data.index,
)
reference_df.insert(
    0,
    seller_column,
    indicator_data[seller_column].values,
)

display(reference_df.head())


,seller_name,environmental_keyword_count,sustainable_material_count,eco_label_count,product_quality_complaint_count,product_damage_complaint_count,product_safety_complaint_count,customer_service_complaint_count,counterfeit_complaint_count,governance_keyword_count
0,101Dealz,0.0,0.555556,0.0,0.0,0.066667,0.333333,0.0,0.0,0.5
1,123(EXOPRT WAREHOUSE),0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0
2,24 Caret,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.1
3,"33,000ft outdoor",0.0,0.333333,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0
4,5Mayi Official,0.0,0.333333,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0


In [43]:
# ============================================================
# REVERSE NEGATIVE INDICATORS
# ============================================================

for feature in negative_direction_features:
    reference_df[feature] = 1.0 - reference_df[feature]

print("Negative features reversed:")
for feature in negative_direction_features:
    print("-", feature)


Negative features reversed:
- product_quality_complaint_count
- product_damage_complaint_count
- product_safety_complaint_count
- customer_service_complaint_count
- counterfeit_complaint_count


In [44]:
# ============================================================
# CALCULATE E, S, G AND OVERALL ESG REFERENCE SCORE
# ============================================================

for dimension_code in ["E", "S", "G"]:
    dimension_features = weight_df.loc[
        weight_df["dimension_code"] == dimension_code,
        "feature",
    ].tolist()

    reference_df[f"{dimension_code}_score"] = sum(
        reference_df[feature]
        * weight_df.set_index("feature").loc[
            feature,
            "local_weight_within_dimension",
        ]
        for feature in dimension_features
    )

reference_df["ESG_reference_score"] = sum(
    reference_df[feature] * AHP_GLOBAL_WEIGHTS[feature]
    for feature in esg_features
)

reference_df["ESG_reference_score_100"] = (
    reference_df["ESG_reference_score"] * 100
)

# Kiểm tra công thức tổng hợp
dimension_weights = (
    dimension_weight_df.set_index("dimension_code")[
        "dimension_weight"
    ].to_dict()
)

dimension_reconstruction = (
    reference_df["E_score"] * dimension_weights["E"]
    + reference_df["S_score"] * dimension_weights["S"]
    + reference_df["G_score"] * dimension_weights["G"]
)

if not np.allclose(
    dimension_reconstruction,
    reference_df["ESG_reference_score"],
    atol=1e-10,
):
    raise ValueError("Điểm E/S/G không khớp điểm ESG tổng hợp.")

reference_df["ESG_rank"] = (
    reference_df["ESG_reference_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

reference_df.to_csv(
    REFERENCE_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(
    reference_df[
        [
            seller_column,
            "E_score",
            "S_score",
            "G_score",
            "ESG_reference_score_100",
            "ESG_rank",
        ]
    ].sort_values("ESG_rank").head(20)
)

,seller_name,E_score,S_score,G_score,ESG_reference_score_100,ESG_rank
38,CHOUYATOU,0.633281,1.00000,0.843725,75.264474,1
87,HOEREV®,0.633281,1.00000,0.843725,75.264474,1
167,QINSEN,0.633281,1.00000,0.804656,74.975002,2
24,Aulemen,0.633281,0.89364,1.000000,73.412659,3
8,AMASALES20,0.471591,1.00000,1.000000,66.026731,4
47,Cupshe Apparel,0.422188,1.00000,1.000000,62.850382,5
232,YUCHENGFU,0.422188,1.00000,1.000000,62.850382,5
26,Avanova,0.422188,1.00000,1.000000,62.850382,5
5,ABAFIP Store,0.422188,1.00000,1.000000,62.850382,5
155,PURE CHAMP,0.422188,1.00000,1.000000,62.850382,5


In [45]:
# ============================================================
# PREPARE X AND y
# ============================================================

X = reference_df[esg_features].copy()
y = reference_df["ESG_reference_score"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("X shape:", X.shape)
print("Train:", X_train.shape)
print("Test:", X_test.shape)


X shape: (261, 9)
Train: (208, 9)
Test: (53, 9)


In [46]:
# ============================================================
# RIDGE GRID SEARCH
# ============================================================

ridge_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", Ridge()),
    ]
)

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={
        "model__alpha": [
            0.001,
            0.01,
            0.1,
            1.0,
            10.0,
            100.0,
        ]
    },
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1,
    return_train_score=True,
)

ridge_search.fit(X_train, y_train)

best_ridge_model = ridge_search.best_estimator_
best_ridge_alpha = ridge_search.best_params_["model__alpha"]

print("Best Ridge alpha:", best_ridge_alpha)
print("Best Ridge CV MSE:", -ridge_search.best_score_)


Best Ridge alpha: 0.001
Best Ridge CV MSE: 0.0001369108644357919


In [47]:
# ============================================================
# BASE ESTIMATORS
# ============================================================

xgboost_estimator = XGBRegressor(
    n_estimators=300,
    learning_rate=0.02,
    max_depth=2,
    min_child_weight=5,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=5.0,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)


In [48]:
# ============================================================
# VOTING WEIGHT GRID SEARCH
# ============================================================

voting_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            VotingRegressor(
                estimators=[
                    (
                        "ridge",
                        Ridge(alpha=best_ridge_alpha),
                    ),
                    (
                        "xgboost",
                        clone(xgboost_estimator),
                    ),
                ],
                n_jobs=-1,
            ),
        ),
    ]
)

voting_weight_search = GridSearchCV(
    estimator=voting_pipeline,
    param_grid={
        "model__weights": [
            [1, 1],
            [2, 1],
            [3, 1],
            [4, 1],
            [5, 1],
            [9, 1],
            [1, 2],
            [1, 3],
            [1, 4],
        ]
    },
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1,
    refit=True,
)

voting_weight_search.fit(X_train, y_train)

best_voting_model = voting_weight_search.best_estimator_
best_voting_weights = voting_weight_search.best_params_[
    "model__weights"
]

print("Best Voting weights [Ridge, XGBoost]:", best_voting_weights)


Best Voting weights [Ridge, XGBoost]: [9, 1]


In [49]:
# ============================================================
# FIVE MODELS
# ============================================================

stacking_estimator = StackingRegressor(
    estimators=[
        (
            "ridge",
            Ridge(alpha=best_ridge_alpha),
        ),
        (
            "xgboost",
            clone(xgboost_estimator),
        ),
    ],
    final_estimator=Ridge(alpha=best_ridge_alpha),
    cv=5,
    passthrough=True,
    n_jobs=-1,
)

models = {
    "Linear Regression": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LinearRegression()),
        ]
    ),
    "Ridge Regression": best_ridge_model,
    "XGBoost Regressor": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", clone(xgboost_estimator)),
        ]
    ),
    "Voting Regressor": best_voting_model,
    "Stacking Regressor": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", stacking_estimator),
        ]
    ),
}

print("Models:")
for name in models:
    print("-", name)


Models:
- Linear Regression
- Ridge Regression
- XGBoost Regressor
- Voting Regressor
- Stacking Regressor


In [50]:
def evaluate_regression(
    model,
    X_train,
    X_test,
    y_train,
    y_test,
):
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    train_pred = fitted_model.predict(X_train)
    test_pred = fitted_model.predict(X_test)

    metrics = {
        "train_MAE": mean_absolute_error(y_train, train_pred),
        "test_MAE": mean_absolute_error(y_test, test_pred),
        "train_MSE": mean_squared_error(y_train, train_pred),
        "test_MSE": mean_squared_error(y_test, test_pred),
        "train_RMSE": np.sqrt(
            mean_squared_error(y_train, train_pred)
        ),
        "test_RMSE": np.sqrt(
            mean_squared_error(y_test, test_pred)
        ),
        "train_R2": r2_score(y_train, train_pred),
        "test_R2": r2_score(y_test, test_pred),
    }

    return fitted_model, metrics


In [51]:
# ============================================================
# TRAIN/TEST EVALUATION
# ============================================================

performance_records = []
trained_models = {}

for model_name, model in models.items():
    fitted_model, metrics = evaluate_regression(
        model,
        X_train,
        X_test,
        y_train,
        y_test,
    )

    trained_models[model_name] = fitted_model
    performance_records.append(
        {
            "model": model_name,
            **metrics,
        }
    )

model_performance_df = pd.DataFrame(performance_records)


In [52]:
# ============================================================
# 5-FOLD CROSS-VALIDATION
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2",
}

for index, row in model_performance_df.iterrows():
    model_name = row["model"]

    cv_result = cross_validate(
        clone(models[model_name]),
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )

    cv_mae = -cv_result["test_MAE"]
    cv_mse = -cv_result["test_MSE"]
    cv_rmse = np.sqrt(cv_mse)
    cv_r2 = cv_result["test_R2"]

    model_performance_df.loc[index, "CV_MAE_mean"] = cv_mae.mean()
    model_performance_df.loc[index, "CV_MAE_std"] = cv_mae.std()
    model_performance_df.loc[index, "CV_MSE_mean"] = cv_mse.mean()
    model_performance_df.loc[index, "CV_MSE_std"] = cv_mse.std()
    model_performance_df.loc[index, "CV_RMSE_mean"] = cv_rmse.mean()
    model_performance_df.loc[index, "CV_RMSE_std"] = cv_rmse.std()
    model_performance_df.loc[index, "CV_R2_mean"] = cv_r2.mean()
    model_performance_df.loc[index, "CV_R2_std"] = cv_r2.std()

model_performance_df = (
    model_performance_df
    .sort_values("CV_RMSE_mean")
    .reset_index(drop=True)
)

display(model_performance_df)

model_performance_df.to_csv(
    PERFORMANCE_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)


,model,train_MAE,test_MAE,train_MSE,test_MSE,train_RMSE,test_RMSE,train_R2,test_R2,CV_MAE_mean,CV_MAE_std,CV_MSE_mean,CV_MSE_std,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std
0,Linear Regression,6.992270e-17,7.331661e-17,9.362982e-33,1.302365e-32,9.676251e-17,1.141212e-16,1.000000,1.000000,1.292910e-16,1.160226e-16,6.928171e-31,1.365749e-30,4.477545e-16,7.016645e-16,1.000000,0.000000
1,Stacking Regressor,1.598818e-04,5.967314e-05,3.883693e-06,1.928326e-08,1.970709e-03,1.388642e-04,0.999572,0.999998,4.011934e-04,7.339828e-04,3.566056e-05,7.129995e-05,2.747141e-03,5.302243e-03,0.995173,0.009652
2,Ridge Regression,1.087882e-05,1.508047e-05,4.480227e-10,1.382008e-09,2.116655e-05,3.717537e-05,1.000000,1.000000,4.232666e-04,8.243194e-04,4.397087e-05,8.794082e-05,2.980211e-03,5.923615e-03,0.994048,0.011905
3,Voting Regressor,7.549262e-04,1.563130e-03,4.524727e-06,1.449520e-05,2.127141e-03,3.807256e-03,0.999502,0.998357,1.430033e-03,8.736451e-04,5.542159e-05,9.818216e-05,5.068764e-03,5.452451e-03,0.992637,0.013358
4,XGBoost Regressor,7.398188e-03,1.548210e-02,4.492461e-04,1.437369e-03,2.119543e-02,3.791265e-02,0.950524,0.837082,1.051889e-02,3.018457e-03,6.803833e-04,4.279644e-04,2.479761e-02,8.090859e-03,0.921699,0.052259


In [53]:
# ============================================================
# SELECT AND REFIT BEST CANDIDATE MODEL
# ============================================================
# Linear được giữ làm reconstruction baseline.
# Khi chọn mô hình triển khai, ưu tiên nhóm candidate không gồm Linear.

candidate_performance_df = model_performance_df[
    model_performance_df["model"] != "Linear Regression"
].copy()

best_model_name = (
    candidate_performance_df
    .sort_values("CV_RMSE_mean")
    .iloc[0]["model"]
)

final_model = clone(models[best_model_name])
final_model.fit(X, y)

print("Best candidate model:", best_model_name)


Best candidate model: Stacking Regressor


In [54]:
# ============================================================
# PERMUTATION IMPORTANCE
# ============================================================

permutation_result = permutation_importance(
    final_model,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

feature_importance_df = pd.DataFrame(
    {
        "feature": esg_features,
        "importance_mean": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std,
    }
).merge(
    weight_df[
        [
            "feature",
            "indicator",
            "dimension",
            "dimension_code",
            "global_weight",
        ]
    ],
    on="feature",
    how="left",
)

positive_importance = feature_importance_df[
    "importance_mean"
].clip(lower=0)

importance_sum = positive_importance.sum()

feature_importance_df["importance_normalized"] = (
    positive_importance / importance_sum
    if importance_sum > 0
    else 0.0
)

feature_importance_df = (
    feature_importance_df
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

feature_importance_df["importance_rank"] = np.arange(
    1,
    len(feature_importance_df) + 1,
)

display(feature_importance_df)

feature_importance_df.to_csv(
    IMPORTANCE_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)


,feature,importance_mean,importance_std,indicator,dimension,dimension_code,global_weight,importance_normalized,importance_rank
0,sustainable_material_count,0.130575,0.004913,Sustainable Material Adoption,Environmental,E,0.407159,0.673912,1
1,product_safety_complaint_count,0.020075,0.001261,Product Safety Responsibility,Social,S,0.179182,0.103607,2
2,eco_label_count,0.014191,0.000034,Eco Label Adoption,Environmental,E,0.167483,0.073239,3
3,product_quality_complaint_count,0.013698,0.000409,Product Quality Responsibility,Social,S,0.073693,0.070699,4
4,governance_keyword_count,0.008156,0.000257,Transparency & Responsibility,Governance,G,0.019298,0.042092,5
5,counterfeit_complaint_count,0.004663,0.000129,Business Integrity,Governance,G,0.046895,0.024068,6
6,product_damage_complaint_count,0.002096,0.000162,Product Durability,Social,S,0.030097,0.010818,7
7,customer_service_complaint_count,0.000303,0.000016,Customer Relationship Management,Governance,G,0.007899,0.001566,8
8,environmental_keyword_count,0.000000,0.000000,Environmental Commitment,Environmental,E,0.068293,0.000000,9


In [55]:
# ============================================================
# SAVE MODEL PACKAGE AND METADATA
# ============================================================

model_package = {
    "model_name": best_model_name,
    "model": final_model,
    "feature_names": esg_features,
    "seller_column": seller_column,
    "reference_imputer": imputer,
    "reference_scaler": scaler,
    "indicator_mapping": indicator_mapping,
    "global_weights": AHP_GLOBAL_WEIGHTS,
    "raw_global_weights": AHP_GLOBAL_WEIGHTS_INPUT,
    "negative_direction_features": negative_direction_features,
    "random_state": RANDOM_STATE,
}

with open(MODEL_OUTPUT_PATH, "wb") as file:
    pickle.dump(model_package, file)

metadata = {
    "notebook": "Notebook 06 - Seller ESG Modeling",
    "target": "ESG_reference_score",
    "target_type": "AHP-derived internal reference score",
    "features": esg_features,
    "number_of_features": len(esg_features),
    "number_of_sellers": len(X),
    "best_candidate_model": best_model_name,
    "linear_role": "reconstruction baseline",
    "best_voting_weights": best_voting_weights,
    "random_state": RANDOM_STATE,
    "output_files": [
        REFERENCE_OUTPUT_PATH.name,
        WEIGHT_OUTPUT_PATH.name,
        PERFORMANCE_OUTPUT_PATH.name,
        IMPORTANCE_OUTPUT_PATH.name,
        MODEL_OUTPUT_PATH.name,
    ],
    "within_dimension_aggregation":
    "equal arithmetic mean",
}

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("Saved:", MODEL_OUTPUT_PATH)
print("Saved:", METADATA_OUTPUT_PATH)


Saved: d:\UNI\NCKH\CTD2026_DT049\ESG_Modeling\seller_esg_model.pkl
Saved: d:\UNI\NCKH\CTD2026_DT049\ESG_Modeling\model_metadata.json


In [56]:
output_summary = []

for output_file in sorted(OUTPUT_DIR.iterdir()):
    if output_file.is_file():
        output_summary.append(
            {
                "file": output_file.name,
                "size_kb": round(
                    output_file.stat().st_size / 1024,
                    2,
                ),
            }
        )

display(pd.DataFrame(output_summary))


,file,size_kb
0,ahp_consistency_results.csv,0.41
1,ahp_fixed_weights.csv,1.39
2,ahp_indicator_mapping.csv,0.64
3,feature_importance.csv,1.42
4,indicator_weight.csv,1.18
5,model_metadata.json,1.03
6,model_performance.csv,1.87
7,ridge_best_params.csv,0.07
8,ridge_tuning_results.csv,2.91
9,seller_esg_model.pkl,263.44
